# AMPS I · Week 11 — Lab
## Stokes flow: mixed spaces, Taylor–Hood, MINI and an unstable pair

| | |
|---|---|
| **Lecture sections** | Part B (mixed spaces, BCs, pressure constant, Poiseuille, MMS) · Part C (inf-sup, instability, cavity) · Part D (LLM) |
| **Time** | ≈ 80 min |
| **You will** | build Taylor–Hood and MINI spaces · solve Poiseuille (exact for Taylor–Hood) and an MMS · see why the error must not be measured against an interpolant · watch P1–P1 fail in two different ways · compute discrete inf-sup constants · check the symmetry of the Stokes cavity |

$-\mu\Delta\mathbf u+\nabla p=\mathbf f$, $\nabla\cdot\mathbf u=0$ on the unit square, $\mathbf u=\mathbf g$ on $\partial\Omega$, $\mu=1$; the pressure is fixed by pinning one DOF and
compared **after removing the means**.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Vector and mixed spaces  *(lecture §B1)*
DOF counts on $16\times16$ for four pairs, and the difference between a sub-function **view** (`wh.split()`) and a **collapsed** copy.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

msh = mesh.create_unit_square(comm, 16, 16)
for pair in ("TH", "MINI", "P1P1", "P2P0"):
    Ve, Qe = elements(pair)
    W = fem.functionspace(msh, basix.ufl.mixed_element([Ve, Qe]))
    V0, _ = W.sub(0).collapse(); Q0, _ = W.sub(1).collapse()
    nu = V0.dofmap.index_map.size_global * V0.dofmap.index_map_bs
    npr = Q0.dofmap.index_map.size_global
    say(f"{pair:5s}: velocity {nu:5d} + pressure {npr:4d} = {W.dofmap.index_map.size_global:5d} DOFs (ratio u/p = {nu / npr:.1f})")
W = fem.functionspace(msh, basix.ufl.mixed_element(list(elements("TH"))))
wh = fem.Function(W)
uview, pview = wh.split()
pcopy = wh.sub(1).collapse()
say(f"view:      len(p.x.array) = {comm.allreduce(pview.x.array.size, op=MPI.SUM)} (the whole mixed vector, incl. ghosts)")
say(f"collapsed: len(p.x.array) = {comm.allreduce(pcopy.x.array.size, op=MPI.SUM)} (pressure only, incl. ghosts)")

## 2 · Plane Poiseuille flow: an exactness test  *(lecture §B3)*
$\mathbf u=(y(1-y),0)$, $p=-2x$ — quadratic velocity and linear pressure lie **in** the Taylor–Hood space, so the discrete solution must be exact up to
round-off. Read `stokes` in the cell first (mixed space, BCs on `W.sub(0)`, one pinned pressure DOF, MUMPS).

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

def stokes(N, pair, diag="right", mu=1.0, u_ex=None, p_ex=None, f=None, u_bc=None, cm=comm):
    """Mixed Stokes solve (mu Lap u form), Dirichlet velocity on the whole boundary, one pressure DOF pinned to 0.
    u_ex, p_ex: functions of the SpatialCoordinate (UFL); f defaults to -mu Lap u_ex + grad p_ex; f="zero" for no force.
    u_bc: a numpy callable for the boundary velocity (default: interpolate u_ex). Returns msh, uh, ph, KSP reason."""
    msh = mesh.create_unit_square(cm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    Ve, Qe = elements(pair)
    W = fem.functionspace(msh, basix.ufl.mixed_element([Ve, Qe]))
    V0, _ = W.sub(0).collapse(); Q0, _ = W.sub(1).collapse()
    x = ufl.SpatialCoordinate(msh)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = fem.form((mu * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx)
    if f == "zero":
        fx = fem.Constant(msh, PETSc.ScalarType((0.0, 0.0)))
    else:
        fx = f(x) if f is not None else -mu * ufl.div(ufl.grad(u_ex(x))) + ufl.grad(p_ex(x))
    L = fem.form(ufl.inner(fx, v) * ufl.dx)
    ub = fem.Function(V0)
    if u_bc is not None:
        ub.interpolate(u_bc)
    else:
        ub.interpolate(fem.Expression(u_ex(x), V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    facets = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(ub, fem.locate_dofs_topological((W.sub(0), V0), 1, facets), W.sub(0))]
    if Qe.degree == 0:                                          # DG0 pressure: pin the value of one cell at the origin
        cells = mesh.locate_entities(msh, 2, lambda X: (X[0] < 1.0 / N + 1e-12) & (X[1] < 1.0 / N + 1e-12))
        cells = cells[cells < msh.topology.index_map(2).size_local][:1]
        msh.topology.create_connectivity(2, 2)
        pdofs = fem.locate_dofs_topological((W.sub(1), Q0), 2, cells)
    else:                                                       # P1 pressure: pin the vertex at the origin
        pdofs = fem.locate_dofs_geometrical((W.sub(1), Q0), lambda X: np.isclose(X[0], 0) & np.isclose(X[1], 0))
    bcs.append(fem.dirichletbc(fem.Function(Q0), pdofs, W.sub(1)))
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L); apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
    ksp = PETSc.KSP().create(cm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    wh = fem.Function(W); ksp.solve(b, wh.x.petsc_vec); wh.x.scatter_forward()
    reason = ksp.getConvergedReason()
    ksp.destroy(); A.destroy(); b.destroy()
    return msh, wh.sub(0).collapse(), wh.sub(1).collapse(), reason

def errors(msh, uh, ph, u_ex, p_ex, cm=comm):
    """[|u-u_h|_L2, |grad(u-u_h)|_L2, |p-p_h|_L2 with both means removed, |div u_h|_L2]; nan if u_h, p_h are not finite.
    u_ex: a function of x (UFL) or a fem.Function."""
    S = lambda e: cm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    if not cm.allreduce(int(np.all(np.isfinite(uh.x.array)) and np.all(np.isfinite(ph.x.array))), op=MPI.MIN):
        return [np.nan] * 4                                     # never put inf/nan into a UFL form (JIT error)
    x = ufl.SpatialCoordinate(msh)
    pm, pem = S(ph), S(p_ex(x))
    du = uh - u_ex if isinstance(u_ex, fem.Function) else uh - u_ex(x)
    return [np.sqrt(S(ufl.inner(du, du))), np.sqrt(S(ufl.inner(ufl.grad(du), ufl.grad(du)))),
            np.sqrt(S(((ph - pm) - (p_ex(x) - pem)) ** 2)), np.sqrt(S(ufl.div(uh) ** 2))]

U_P = lambda x: ufl.as_vector([x[1] * (1 - x[1]), x[1] * 0])
P_P = lambda x: -2 * x[0]

msh, uh, ph, reason = stokes(8, "TH", u_ex=U_P, p_ex=P_P)
e = errors(msh, uh, ph, U_P, P_P)
say(f"Taylor-Hood, N = 8: KSP reason {reason}; |u-u_h|_L2 = {e[0]:.1e}, |grad(u-u_h)| = {e[1]:.1e}, |p-p_h| = {e[2]:.1e}, |div u_h| = {e[3]:.1e}")

> **✏️ Your turn (5 min).** Solve the same with `"MINI"` and `"P2P0"`. Are they exact? Explain from the spaces which part of $(\mathbf u,p)$ each pair cannot represent.

In [ ]:
%%fenicsx -np 2
# TODO: Poiseuille with MINI and P2P0

## 3 · MMS: Taylor–Hood rates  *(lecture §A6, §B3–B4)*
$\tilde{\mathbf u}=(\sin\pi x\cos\pi y,\,-\cos\pi x\sin\pi y)$ (check: divergence-free), $\tilde p=\sin\pi x\sin\pi y$, $N=8,\dots,64$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

def stokes(N, pair, diag="right", mu=1.0, u_ex=None, p_ex=None, f=None, u_bc=None, cm=comm):
    """Mixed Stokes solve (mu Lap u form), Dirichlet velocity on the whole boundary, one pressure DOF pinned to 0.
    u_ex, p_ex: functions of the SpatialCoordinate (UFL); f defaults to -mu Lap u_ex + grad p_ex; f="zero" for no force.
    u_bc: a numpy callable for the boundary velocity (default: interpolate u_ex). Returns msh, uh, ph, KSP reason."""
    msh = mesh.create_unit_square(cm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    Ve, Qe = elements(pair)
    W = fem.functionspace(msh, basix.ufl.mixed_element([Ve, Qe]))
    V0, _ = W.sub(0).collapse(); Q0, _ = W.sub(1).collapse()
    x = ufl.SpatialCoordinate(msh)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = fem.form((mu * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx)
    if f == "zero":
        fx = fem.Constant(msh, PETSc.ScalarType((0.0, 0.0)))
    else:
        fx = f(x) if f is not None else -mu * ufl.div(ufl.grad(u_ex(x))) + ufl.grad(p_ex(x))
    L = fem.form(ufl.inner(fx, v) * ufl.dx)
    ub = fem.Function(V0)
    if u_bc is not None:
        ub.interpolate(u_bc)
    else:
        ub.interpolate(fem.Expression(u_ex(x), V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    facets = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(ub, fem.locate_dofs_topological((W.sub(0), V0), 1, facets), W.sub(0))]
    if Qe.degree == 0:                                          # DG0 pressure: pin the value of one cell at the origin
        cells = mesh.locate_entities(msh, 2, lambda X: (X[0] < 1.0 / N + 1e-12) & (X[1] < 1.0 / N + 1e-12))
        cells = cells[cells < msh.topology.index_map(2).size_local][:1]
        msh.topology.create_connectivity(2, 2)
        pdofs = fem.locate_dofs_topological((W.sub(1), Q0), 2, cells)
    else:                                                       # P1 pressure: pin the vertex at the origin
        pdofs = fem.locate_dofs_geometrical((W.sub(1), Q0), lambda X: np.isclose(X[0], 0) & np.isclose(X[1], 0))
    bcs.append(fem.dirichletbc(fem.Function(Q0), pdofs, W.sub(1)))
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L); apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
    ksp = PETSc.KSP().create(cm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    wh = fem.Function(W); ksp.solve(b, wh.x.petsc_vec); wh.x.scatter_forward()
    reason = ksp.getConvergedReason()
    ksp.destroy(); A.destroy(); b.destroy()
    return msh, wh.sub(0).collapse(), wh.sub(1).collapse(), reason

def errors(msh, uh, ph, u_ex, p_ex, cm=comm):
    """[|u-u_h|_L2, |grad(u-u_h)|_L2, |p-p_h|_L2 with both means removed, |div u_h|_L2]; nan if u_h, p_h are not finite.
    u_ex: a function of x (UFL) or a fem.Function."""
    S = lambda e: cm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    if not cm.allreduce(int(np.all(np.isfinite(uh.x.array)) and np.all(np.isfinite(ph.x.array))), op=MPI.MIN):
        return [np.nan] * 4                                     # never put inf/nan into a UFL form (JIT error)
    x = ufl.SpatialCoordinate(msh)
    pm, pem = S(ph), S(p_ex(x))
    du = uh - u_ex if isinstance(u_ex, fem.Function) else uh - u_ex(x)
    return [np.sqrt(S(ufl.inner(du, du))), np.sqrt(S(ufl.inner(ufl.grad(du), ufl.grad(du)))),
            np.sqrt(S(((ph - pm) - (p_ex(x) - pem)) ** 2)), np.sqrt(S(ufl.div(uh) ** 2))]

U_EX = lambda x: ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
P_EX = lambda x: ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])

R = []
for N in (8, 16, 32, 64):
    msh, uh, ph, reason = stokes(N, "TH", u_ex=U_EX, p_ex=P_EX)
    R.append(errors(msh, uh, ph, U_EX, P_EX))
R = np.array(R)
for j, name in enumerate(("|u-u_h|_L2", "|grad(u-u_h)|", "|p-p_h|_L2", "|div u_h|")):
    say(f"{name:14s}: {[f'{e:.2e}' for e in R[:, j]]}  rates {rate(R[:, j])}")

> **✏️ Your turn (5 min).** Measure the velocity errors against the P2 **interpolant** $I_h\tilde{\mathbf u}$ instead (pass a `fem.Function` to `errors`).
> Which rates do you get, and why are they not errors?

In [ ]:
%%fenicsx -np 2
# TODO: errors against the interpolant

## 4 · P1–P1: two ways to fail  *(lecture §C2)*
The same MMS with P1–P1 on the `right` and on the `crossed` mesh, $N=16$ (serial). The pressures are saved for a plot together with Taylor–Hood.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

def stokes(N, pair, diag="right", mu=1.0, u_ex=None, p_ex=None, f=None, u_bc=None, cm=comm):
    """Mixed Stokes solve (mu Lap u form), Dirichlet velocity on the whole boundary, one pressure DOF pinned to 0.
    u_ex, p_ex: functions of the SpatialCoordinate (UFL); f defaults to -mu Lap u_ex + grad p_ex; f="zero" for no force.
    u_bc: a numpy callable for the boundary velocity (default: interpolate u_ex). Returns msh, uh, ph, KSP reason."""
    msh = mesh.create_unit_square(cm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    Ve, Qe = elements(pair)
    W = fem.functionspace(msh, basix.ufl.mixed_element([Ve, Qe]))
    V0, _ = W.sub(0).collapse(); Q0, _ = W.sub(1).collapse()
    x = ufl.SpatialCoordinate(msh)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = fem.form((mu * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx)
    if f == "zero":
        fx = fem.Constant(msh, PETSc.ScalarType((0.0, 0.0)))
    else:
        fx = f(x) if f is not None else -mu * ufl.div(ufl.grad(u_ex(x))) + ufl.grad(p_ex(x))
    L = fem.form(ufl.inner(fx, v) * ufl.dx)
    ub = fem.Function(V0)
    if u_bc is not None:
        ub.interpolate(u_bc)
    else:
        ub.interpolate(fem.Expression(u_ex(x), V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    facets = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(ub, fem.locate_dofs_topological((W.sub(0), V0), 1, facets), W.sub(0))]
    if Qe.degree == 0:                                          # DG0 pressure: pin the value of one cell at the origin
        cells = mesh.locate_entities(msh, 2, lambda X: (X[0] < 1.0 / N + 1e-12) & (X[1] < 1.0 / N + 1e-12))
        cells = cells[cells < msh.topology.index_map(2).size_local][:1]
        msh.topology.create_connectivity(2, 2)
        pdofs = fem.locate_dofs_topological((W.sub(1), Q0), 2, cells)
    else:                                                       # P1 pressure: pin the vertex at the origin
        pdofs = fem.locate_dofs_geometrical((W.sub(1), Q0), lambda X: np.isclose(X[0], 0) & np.isclose(X[1], 0))
    bcs.append(fem.dirichletbc(fem.Function(Q0), pdofs, W.sub(1)))
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L); apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
    ksp = PETSc.KSP().create(cm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    wh = fem.Function(W); ksp.solve(b, wh.x.petsc_vec); wh.x.scatter_forward()
    reason = ksp.getConvergedReason()
    ksp.destroy(); A.destroy(); b.destroy()
    return msh, wh.sub(0).collapse(), wh.sub(1).collapse(), reason

def errors(msh, uh, ph, u_ex, p_ex, cm=comm):
    """[|u-u_h|_L2, |grad(u-u_h)|_L2, |p-p_h|_L2 with both means removed, |div u_h|_L2]; nan if u_h, p_h are not finite.
    u_ex: a function of x (UFL) or a fem.Function."""
    S = lambda e: cm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    if not cm.allreduce(int(np.all(np.isfinite(uh.x.array)) and np.all(np.isfinite(ph.x.array))), op=MPI.MIN):
        return [np.nan] * 4                                     # never put inf/nan into a UFL form (JIT error)
    x = ufl.SpatialCoordinate(msh)
    pm, pem = S(ph), S(p_ex(x))
    du = uh - u_ex if isinstance(u_ex, fem.Function) else uh - u_ex(x)
    return [np.sqrt(S(ufl.inner(du, du))), np.sqrt(S(ufl.inner(ufl.grad(du), ufl.grad(du)))),
            np.sqrt(S(((ph - pm) - (p_ex(x) - pem)) ** 2)), np.sqrt(S(ufl.div(uh) ** 2))]

U_EX = lambda x: ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
P_EX = lambda x: ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])

out = {}
for pair, diag in (("TH", "right"), ("P1P1", "right"), ("P1P1", "crossed")):
    msh, uh, ph, reason = stokes(16, pair, diag, u_ex=U_EX, p_ex=P_EX)
    e = errors(msh, uh, ph, U_EX, P_EX)
    finite = np.all(np.isfinite(ph.x.array))
    print(f"{pair:4s} {diag:7s}: KSP reason {reason:3d}; finite: {finite}; |u-u_h|_L2 {e[0]:.2e}, |p-p_h| {e[2]:.2e}" +
          (f", p_h in [{ph.x.array.min():.1f}, {ph.x.array.max():.1f}]" if finite else ""))
    if finite:
        Q = ph.function_space
        out[f"{pair}_{diag}"] = np.c_[Q.tabulate_dof_coordinates()[:, :2], ph.x.array - ph.x.array.mean()]
        out[f"{pair}_{diag}_tri"] = Q.dofmap.list
np.savez("/tmp/fnx/p1p1.npz", **out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/p1p1.npz")
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, key in zip(axs, ("TH_right", "P1P1_crossed")):
    X, tri = d[key], d[key + "_tri"]
    tp = ax.tripcolor(mtri.Triangulation(X[:, 0], X[:, 1], tri), X[:, 2], shading="gouraud")
    fig.colorbar(tp, ax=ax); ax.set_aspect("equal"); ax.set_title(f"p_h: {key}")
fig.tight_layout(); plt.show()

## 5 · Discrete inf-sup constants  *(lecture §A4, §C1)*
$\beta_h^2=\lambda_{\min}(BA^{-1}B^\top,M_p)$ on $L^2_0$; zero eigenvalues beyond the constant are spurious modes. Serial, dense eigenvalues — small meshes only.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

import scipy.linalg as sla
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import dolfinx.fem.petsc as fpetsc

def infsup(N, pair, diag="right"):
    """Serial. beta_h^2 = smallest non-zero eigenvalue of (B A^-1 B^T, M_p); A = vector Laplacian on interior velocity DOFs.
    Returns (beta_h, number of zero eigenvalues incl. the constant, dim Q_h)."""
    msh = mesh.create_unit_square(MPI.COMM_SELF, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    Ve, Qe = elements(pair)
    V, Q = fem.functionspace(msh, Ve), fem.functionspace(msh, Qe)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    p, q = ufl.TrialFunction(Q), ufl.TestFunction(Q)
    A = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_scipy()
    Bp = fpetsc.assemble_matrix(fem.form(-q * ufl.div(u) * ufl.dx)); Bp.assemble()   # rectangular + blocked: PETSc, not fem.assemble_matrix
    ai, aj, av = Bp.getValuesCSR(); B = sp.csr_matrix((av, aj, ai), shape=Bp.getSize()); Bp.destroy()
    Mp = fem.assemble_matrix(fem.form(p * q * ufl.dx)).to_dense()
    msh.topology.create_connectivity(1, 2)
    bd = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    bs = V.dofmap.index_map_bs
    I = np.setdiff1d(np.arange(A.shape[0]), np.concatenate([bs * bd + c for c in range(bs)]))
    A = A.tocsr()[I][:, I].tocsc(); B = B.tocsr()[:, I]
    S = B @ spla.splu(A).solve(B.T.toarray()); S = 0.5 * (S + S.T)
    ev = sla.eigh(S, Mp, eigvals_only=True)
    nz = int(np.sum(ev < 1e-10 * ev.max()))
    return np.sqrt(ev[nz]), nz, len(ev)

for pair in ("TH", "P1P1"):
    row = []
    for N in (4, 8, 16):
        b, nz, n = infsup(N, pair, "right")
        row.append(f"N={N}: beta_h {b:.3f}, zero modes {nz}/{n}")
    print(f"{pair:5s} " + " | ".join(row), flush=True)

> **✏️ Your turn (10 min).** Compute $\beta_h$ and the zero modes for **P2–P0** and **P1–P0** on the `right` mesh, $N=4,8,16$. Which is stable? Explain the P1–P0
> numbers with a DOF count (interior velocity DOFs vs pressure DOFs).

In [ ]:
%%fenicsx -np 1
# TODO: inf-sup for P2-P0 and P1-P0

## 6 · The Stokes cavity and its symmetry  *(lecture §C5)*
Lid $\mathbf u=(1,0)$ on $y=1$, Taylor–Hood, $N=32$; stream function $-\Delta\psi=\partial_xu_y-\partial_yu_x$, $\psi=0$ on $\partial\Omega$. Saved to `/tmp/fnx/cavity.npz`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

def stokes(N, pair, diag="right", mu=1.0, u_ex=None, p_ex=None, f=None, u_bc=None, cm=comm):
    """Mixed Stokes solve (mu Lap u form), Dirichlet velocity on the whole boundary, one pressure DOF pinned to 0.
    u_ex, p_ex: functions of the SpatialCoordinate (UFL); f defaults to -mu Lap u_ex + grad p_ex; f="zero" for no force.
    u_bc: a numpy callable for the boundary velocity (default: interpolate u_ex). Returns msh, uh, ph, KSP reason."""
    msh = mesh.create_unit_square(cm, N, N, diagonal=getattr(mesh.DiagonalType, diag))
    Ve, Qe = elements(pair)
    W = fem.functionspace(msh, basix.ufl.mixed_element([Ve, Qe]))
    V0, _ = W.sub(0).collapse(); Q0, _ = W.sub(1).collapse()
    x = ufl.SpatialCoordinate(msh)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = fem.form((mu * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx)
    if f == "zero":
        fx = fem.Constant(msh, PETSc.ScalarType((0.0, 0.0)))
    else:
        fx = f(x) if f is not None else -mu * ufl.div(ufl.grad(u_ex(x))) + ufl.grad(p_ex(x))
    L = fem.form(ufl.inner(fx, v) * ufl.dx)
    ub = fem.Function(V0)
    if u_bc is not None:
        ub.interpolate(u_bc)
    else:
        ub.interpolate(fem.Expression(u_ex(x), V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    facets = mesh.exterior_facet_indices(msh.topology)
    bcs = [fem.dirichletbc(ub, fem.locate_dofs_topological((W.sub(0), V0), 1, facets), W.sub(0))]
    if Qe.degree == 0:                                          # DG0 pressure: pin the value of one cell at the origin
        cells = mesh.locate_entities(msh, 2, lambda X: (X[0] < 1.0 / N + 1e-12) & (X[1] < 1.0 / N + 1e-12))
        cells = cells[cells < msh.topology.index_map(2).size_local][:1]
        msh.topology.create_connectivity(2, 2)
        pdofs = fem.locate_dofs_topological((W.sub(1), Q0), 2, cells)
    else:                                                       # P1 pressure: pin the vertex at the origin
        pdofs = fem.locate_dofs_geometrical((W.sub(1), Q0), lambda X: np.isclose(X[0], 0) & np.isclose(X[1], 0))
    bcs.append(fem.dirichletbc(fem.Function(Q0), pdofs, W.sub(1)))
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L); apply_lifting(b, [a], [bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, bcs)
    ksp = PETSc.KSP().create(cm); ksp.setOperators(A); ksp.setType("preonly")
    ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    wh = fem.Function(W); ksp.solve(b, wh.x.petsc_vec); wh.x.scatter_forward()
    reason = ksp.getConvergedReason()
    ksp.destroy(); A.destroy(); b.destroy()
    return msh, wh.sub(0).collapse(), wh.sub(1).collapse(), reason

def errors(msh, uh, ph, u_ex, p_ex, cm=comm):
    """[|u-u_h|_L2, |grad(u-u_h)|_L2, |p-p_h|_L2 with both means removed, |div u_h|_L2]; nan if u_h, p_h are not finite.
    u_ex: a function of x (UFL) or a fem.Function."""
    S = lambda e: cm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    if not cm.allreduce(int(np.all(np.isfinite(uh.x.array)) and np.all(np.isfinite(ph.x.array))), op=MPI.MIN):
        return [np.nan] * 4                                     # never put inf/nan into a UFL form (JIT error)
    x = ufl.SpatialCoordinate(msh)
    pm, pem = S(ph), S(p_ex(x))
    du = uh - u_ex if isinstance(u_ex, fem.Function) else uh - u_ex(x)
    return [np.sqrt(S(ufl.inner(du, du))), np.sqrt(S(ufl.inner(ufl.grad(du), ufl.grad(du)))),
            np.sqrt(S(((ph - pm) - (p_ex(x) - pem)) ** 2)), np.sqrt(S(ufl.div(uh) ** 2))]

from dolfinx.fem.petsc import LinearProblem

def cavity(N, diag="right"):
    """Stokes cavity with Taylor-Hood; returns (P2 DOF coordinates, psi values) gathered on rank 0."""
    msh, uh, ph, reason = stokes(N, "TH", diag, f="zero",
                                 u_bc=lambda X: np.vstack([np.where(np.isclose(X[1], 1.0), 1.0, 0.0), np.zeros(X.shape[1])]))
    Vs = fem.functionspace(msh, ("Lagrange", 2))
    psi, phi = ufl.TrialFunction(Vs), ufl.TestFunction(Vs)
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(Vs, 1, mesh.exterior_facet_indices(msh.topology)), Vs)
    psih = LinearProblem(ufl.inner(ufl.grad(psi), ufl.grad(phi)) * ufl.dx, (uh[1].dx(0) - uh[0].dx(1)) * phi * ufl.dx, bcs=[bc],
                         petsc_options_prefix=f"psi{N}_", petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    n = Vs.dofmap.index_map.size_local
    data = comm.gather(np.c_[Vs.tabulate_dof_coordinates()[:n, :2], psih.x.array[:n]], root=0)
    return np.vstack(data) if comm.rank == 0 else None

d = cavity(32)
if comm.rank == 0:
    i = np.argmin(d[:, 2])
    print(f"min psi = {d[i, 2]:.5f} at ({d[i, 0]:.4f}, {d[i, 1]:.4f})")
    np.save("/tmp/fnx/cavity.npy", d)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/cavity.npy")
fig, ax = plt.subplots(figsize=(5, 5))
ax.tricontour(mtri.Triangulation(d[:, 0], d[:, 1]), d[:, 2], levels=np.linspace(d[:, 2].min(), 0, 14)[:-1], colors="C0")
ax.axvline(0.5, color="0.6", ls=":"); ax.set_aspect("equal"); ax.set_title("Stokes cavity: streamlines")
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** Stokes flow is reversible, so $\psi(x,y)=\psi(1-x,y)$. Measure $\max|\psi(x,y)-\psi(1-x,y)|$ over the DOFs for $N=16$ and
> $N=32$ (the P2 DOF set of the `right` mesh is mirror-symmetric; match points with a tolerance). Is the discrete solution exactly symmetric? Why (not)?

In [ ]:
%%fenicsx -np 2
# TODO: symmetry defect of the cavity solution for N = 16 and 32

## 7 · LLM-assisted coding: live demo  *(lecture §D1)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Solve the Stokes equations on the unit square with Taylor-Hood (P2-P1) elements
using basix.ufl.mixed_element. Manufactured solution u = (sin(pi x) cos(pi y), -cos(pi x) sin(pi y)), p = sin(pi x) sin(pi y), mu = 1,
Dirichlet velocity on the whole boundary. Fix the pressure constant. Print the L2 velocity error, the H1 velocity error and the L2 pressure
error (with the mean removed) for N = 8, 16, 32 on rank 0, with rates. Use MUMPS.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] `basix.ufl.element(..., shape=(2,))` and `mixed_element` — no `VectorElement`, `MixedElement`, `VectorFunctionSpace` (removed)
- [ ] BCs through `locate_dofs_*((W.sub(0), V0), ...)` and `dirichletbc(u_bc, dofs, W.sub(0))`
- [ ] The pressure constant is fixed (pin or nullspace); the pressure error is computed **without** the means
- [ ] Errors against the exact UFL expression, not an interpolant: rates 3 / 2 / 2 (not 4 / 3)
- [ ] The KSP reason is printed; the same numbers with `-np 1`